# Ensemble QK-LSTM — Colab notebook

**Before running:** Runtime → Change runtime type → GPU (T4). Put the Kaggle CSVs (`BID.csv`, `VCB.csv`, …, and the VN-Index file named `VNINDEX.csv`) in one Drive folder and set `DATA_DIR` in the next cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['QK_DATA_DIR']='/content/drive/MyDrive/vn30'      # <- your folder of CSVs
os.environ['QK_OUT_DIR']='/content/drive/MyDrive/vn30/data'   # <- results land here (copy next to the HTML)
# os.environ['QK_FAST']='1'      # uncomment for a quick smoke test
# os.environ['QK_ONLY']='BID'    # uncomment to run a single stock first
!pip -q install pennylane

# Ensemble QK-LSTM — training, evaluation and export for the prototype

Produces, for every stock CSV found in `DATA_DIR`:
  * Classical LSTM (30-40-50-60, ReLU, dropout 0.1)
  * QK-LSTM (same architecture, gate transforms replaced by quantum-kernel sums)
  * Stacking ensemble (Ridge meta-learner on out-of-fold predictions)
  * Persistence ("tomorrow = today") baseline
  * MAE / RMSE / MAPE / R2 / accuracy and Diebold-Mariano tests
and writes `data/<CODE>.js` + `data/<CODE>.json` + `data/manifest.js` for the HTML.

Leakage rules enforced in code:
  1. MinMax scaler is fit on TRAIN rows only (date <= TRAIN_END).
  2. Indicators (SMA/RSI/MACD) are causal: value at day t uses data up to day t.
  3. Target = close of day t+1; the window is days t-59..t.
  4. Test windows reach back into the training period for context (no gaps).
  5. Early stopping watches a chronological validation tail of the TRAIN set only.
  6. Ridge is fit on out-of-fold predictions only.

In [ ]:
import os, json, time, math, glob, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit

# ----------------------------- CONFIG ---------------------------------------
DATA_DIR  = os.environ.get("QK_DATA_DIR", "/content/drive/MyDrive/vn30")  # folder with <CODE>.csv
OUT_DIR   = os.environ.get("QK_OUT_DIR",  "data")
SEQ       = 60
TRAIN_END, TEST_START, TEST_END = "2020-12-31", "2021-01-01", "2021-04-01"
UNITS     = [30, 40, 50, 60]
ACT       = "relu"          # paper: ReLU. Use "tanh" for the textbook LSTM.
DROPOUT   = 0.1
N_QUBITS  = 4
N_REF     = 8               # N reference vectors per gate
Q_LAYERS  = 2               # encode -> CNOT -> encode ... (>=2 so entanglement affects the kernel)
BATCH     = 32              # paper setting (classical LSTM)
BATCH_QK  = 128             # larger batch for QK-LSTM: the simulator is the bottleneck (documented deviation)
LR        = 1e-3
MAX_EPOCHS, PATIENCE = 1000, 8        # paper settings
N_SPLITS  = 5
SEED      = 42
FAST      = os.environ.get("QK_FAST") == "1"      # quick CPU smoke test
if FAST:
    MAX_EPOCHS, PATIENCE, N_SPLITS = 12, 3, 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NAMES = {"VNINDEX": "VN-Index", "BID": "Bank for Investment and Development of Vietnam", "BVH": "Bao Viet Group",
 "CTG": "Bank of Industry and Trade of Vietnam", "FPT": "FPT Corporation", "GAS": "Vietnam Gas Corporation",
 "HDB": "Ho Chi Minh City Development Commercial Bank", "HPG": "Hoa Phat Group", "KDH": "Khang Dien House",
 "MBB": "Military Commercial Joint Stock Bank", "MSN": "Masan Group", "MWG": "Mobile World Investment",
 "NVL": "Nova Real Estate Investment Group", "PDR": "Phat Dat Real Estate Development", "PLX": "Vietnam National Petroleum",
 "PNJ": "Phu Nhuan Jewelry", "POW": "PetroVietnam Power", "REE": "Refrigeration Electrical Engineering",
 "SBT": "Thanh Thanh Cong - Bien Hoa", "SSI": "SSI Securities", "STB": "Saigon Thuong Tin Commercial Bank",
 "TCB": "Vietnam Technological and Commercial Bank", "TCH": "Hoang Huy Financial Services",
 "TPB": "Tien Phong Commercial Bank", "VCB": "Vietcombank", "VHM": "Vinhomes", "VIC": "Vingroup",
 "VJC": "Vietjet Aviation", "VNM": "Vietnam Dairy Products", "VPB": "Vietnam Prosperity Bank", "VRE": "Vincom Retail"}

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

## 1. Data + technical indicators (SMA10, RSI14, MACD 12/26)

In [ ]:
FEATS = ["open", "high", "low", "close", "volume", "sma10", "rsi14", "macd"]
CLOSE_IDX = FEATS.index("close")

def load_csv(path):
    d = pd.read_csv(path)
    d.columns = [c.strip("<>").lower() for c in d.columns]
    d = d.rename(columns={"dtyyyymmdd": "date", "ticker": "code"})
    d["date"] = pd.to_datetime(d["date"].astype(str), format="%Y%m%d")
    d = d.sort_values("date").drop_duplicates("date").reset_index(drop=True)
    return d[["date", "open", "high", "low", "close", "volume"]]

def add_indicators(d):
    d = d.copy()
    d["sma10"] = d["close"].rolling(10).mean()
    chg = d["close"].diff()
    gain, loss = chg.clip(lower=0).rolling(14).mean(), (-chg.clip(upper=0)).rolling(14).mean()
    d["rsi14"] = 100 - 100 / (1 + gain / loss.replace(0, np.nan))        # simple-average RSI, as in the paper
    d["macd"] = d["close"].ewm(span=12, adjust=False).mean() - d["close"].ewm(span=26, adjust=False).mean()
    return d.dropna().reset_index(drop=True)

def make_samples(d):
    """Return X (N,SEQ,8) scaled, y scaled, target dates, last-close (for persistence), scaler params."""
    train_rows = d["date"] <= TRAIN_END
    lo = d.loc[train_rows, FEATS].min().values; hi = d.loc[train_rows, FEATS].max().values   # TRAIN ONLY
    A = (d[FEATS].values - lo) / np.where(hi - lo == 0, 1, hi - lo)
    X, y, dates, last_close = [], [], [], []
    for i in range(SEQ - 1, len(d) - 1):
        X.append(A[i - SEQ + 1: i + 1]); y.append(A[i + 1, CLOSE_IDX])
        dates.append(d["date"].iloc[i + 1]); last_close.append(d["close"].iloc[i])
    X, y = np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)
    dates = pd.to_datetime(dates); last_close = np.array(last_close)
    tr = dates <= TRAIN_END
    te = (dates >= TEST_START) & (dates <= TEST_END)
    inv = lambda v: v * (hi[CLOSE_IDX] - lo[CLOSE_IDX]) + lo[CLOSE_IDX]
    return X, y, dates, last_close, tr, te, inv

## 2. Models
A shared custom LSTM layer so the ONLY difference between the baseline and QK-LSTM is the gate transform.

**QK gate** (paper Sec. 2.5): z_g = sum_j a_j^(g) * k(v_t, v_j^(g)) + b^(g),
k(v, v_j) = |<phi(v_j)|phi(v)>|^2, with a 4-qubit state-vector circuit simulated in PyTorch
(H -> [Rz,Ry encoding -> CNOT chain] x Q_LAYERS). The projection `v -> angles` and the reference
angles are learned. This fills the gap in the paper: how a ~38-dim v_t becomes 8 rotation angles.

In [ ]:
def _cnot_chain_perm(n):
    """Index permutation applying CNOT(0,1),CNOT(1,2),... to a state vector (qubit 0 = most significant bit)."""
    perm = np.arange(2 ** n)
    def flip(idx):
        bits = [(idx >> (n - 1 - q)) & 1 for q in range(n)]
        for c in range(n - 1):
            if bits[c]: bits[c + 1] ^= 1
        return sum(b << (n - 1 - q) for q, b in enumerate(bits))
    new_pos = np.array([flip(i) for i in range(2 ** n)])
    inv = np.empty_like(new_pos); inv[new_pos] = np.arange(2 ** n)    # new_psi[new_pos[i]] = psi[i]
    return torch.tensor(inv, dtype=torch.long)

class QSim(nn.Module):
    """Differentiable batched state-vector simulator. angles (M, 2n) -> state (M, 2^n) complex."""
    def __init__(self, n=N_QUBITS, layers=Q_LAYERS):
        super().__init__(); self.n, self.layers = n, layers
        self.register_buffer("perm", _cnot_chain_perm(n))
    def forward(self, ang):
        M, n = ang.shape[0], self.n
        theta, phi = ang[:, :n], ang[:, n:]
        c, s = torch.cos(theta / 2), torch.sin(theta / 2)
        e_m, e_p = torch.polar(torch.ones_like(phi), -phi / 2), torch.polar(torch.ones_like(phi), phi / 2)
        # G = Ry(theta) @ Rz(phi), one 2x2 per qubit: (M, n, 2, 2)
        G = torch.stack([torch.stack([c * e_m, -s * e_p], -1), torch.stack([s * e_m, c * e_p], -1)], -2)
        # fuse: K = G_0 (x) G_1 (x) G_2 (x) G_3  (M,16,16); one encode+CNOT layer is  P @ K
        K = torch.einsum("mab,mcd,mef,mgh->macegbdfh", G[:, 0], G[:, 1], G[:, 2], G[:, 3]).reshape(M, 2 ** n, 2 ** n)
        PK = K[:, self.perm, :]                                                           # row-permute = CNOT chain after K
        U = PK
        for _ in range(self.layers - 1): U = torch.bmm(PK, U)
        psi0 = torch.full((M, 2 ** n, 1), 0.5 ** (n / 2), dtype=torch.complex64, device=ang.device)   # H^n |0^n>
        return torch.bmm(U, psi0).squeeze(-1)

class LSTMLayer(nn.Module):
    def __init__(self, in_dim, hid, kind, act=ACT):
        super().__init__()
        self.hid, self.kind = hid, kind
        self.act = torch.relu if act == "relu" else torch.tanh
        d = in_dim + hid
        if kind == "lstm":
            self.lin = nn.Linear(d, 4 * hid)
            with torch.no_grad(): self.lin.bias[hid:2 * hid] = 1.0                        # forget-gate bias
        else:
            self.sim = QSim()
            self.proj = nn.Linear(d, 4 * 2 * N_QUBITS)                                    # v -> 4 gates x 8 angles
            self.ref = nn.Parameter(torch.rand(4, N_REF, 2 * N_QUBITS) * 2 * math.pi - math.pi)
            self.a = nn.Parameter(torch.randn(4, N_REF, hid) * (1.0 / math.sqrt(N_REF)))
            self.b = nn.Parameter(torch.zeros(4, hid)); self.b.data[1] = 1.0              # gate order i,f,g,o
    def gates(self, v, ref_state):
        if self.kind == "lstm":
            return self.lin(v).view(v.shape[0], 4, self.hid)
        B = v.shape[0]
        ang = math.pi * torch.tanh(self.proj(v)).view(B * 4, -1)
        psi = self.sim(ang).view(B, 4, -1)                                                # (B,4,2^n)
        k = torch.einsum("gns,bgs->bgn", ref_state.conj(), psi).abs() ** 2                # fidelity kernel (B,4,N)
        return torch.einsum("bgn,gnh->bgh", k, self.a) + self.b
    def forward(self, x):
        B, T, _ = x.shape
        h = x.new_zeros(B, self.hid); c = x.new_zeros(B, self.hid)
        ref_state = self.sim(math.pi * torch.tanh(self.ref).view(4 * N_REF, -1)).view(4, N_REF, -1) if self.kind == "qk" else None
        out = []
        for t in range(T):
            z = self.gates(torch.cat([h, x[:, t]], -1), ref_state)
            i, f, g, o = torch.sigmoid(z[:, 0]), torch.sigmoid(z[:, 1]), self.act(z[:, 2]), torch.sigmoid(z[:, 3])
            c = f * c + i * g
            h = o * self.act(c)
            out.append(h)
        return torch.stack(out, 1)

class StackedNet(nn.Module):
    def __init__(self, kind, n_feat=len(FEATS)):
        super().__init__()
        dims = [n_feat] + UNITS
        self.layers = nn.ModuleList([LSTMLayer(dims[i], dims[i + 1], kind) for i in range(len(UNITS))])
        self.drop = nn.Dropout(DROPOUT); self.head = nn.Linear(UNITS[-1], 1)
    def forward(self, x):
        for L in self.layers: x = self.drop(L(x))
        return self.head(x[:, -1]).squeeze(-1)

def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 3. Training / prediction helpers

In [ ]:
def fit(kind, Xtr, ytr, max_epochs=MAX_EPOCHS, patience=PATIENCE, val_frac=0.1, verbose=False):
    """Train on Xtr; early-stop on the chronological last val_frac of Xtr (never on test data)."""
    set_seed()
    nv = max(int(len(Xtr) * val_frac), 20)
    Xt, yt, Xv, yv = (torch.tensor(a, device=DEVICE) for a in (Xtr[:-nv], ytr[:-nv], Xtr[-nv:], ytr[-nv:]))
    net = StackedNet(kind).to(DEVICE); opt = torch.optim.Adam(net.parameters(), lr=LR); lossf = nn.MSELoss()
    best, best_state, bad = float("inf"), None, 0
    for ep in range(max_epochs):
        net.train(); perm = torch.randperm(len(Xt), device=DEVICE)
        bs = BATCH_QK if kind == "qk" else BATCH
        for i in range(0, len(Xt), bs):
            idx = perm[i:i + bs]; opt.zero_grad()
            loss = lossf(net(Xt[idx]), yt[idx]); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
        net.eval()
        with torch.no_grad(): vl = lossf(net(Xv), yv).item()
        if verbose: print(f"  [{kind}] epoch {ep+1:3d} val_mse={vl:.6f}")
        if vl < best - 1e-7: best, bad, best_state = vl, 0, {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(best_state); net.eval()
    return net, ep + 1

@torch.no_grad()
def predict(net, X):
    out = [net(torch.tensor(X[i:i + 256], device=DEVICE)).cpu().numpy() for i in range(0, len(X), 256)]
    return np.concatenate(out)

def metrics(y, p):
    e = y - p
    mae, rmse = np.mean(np.abs(e)), np.sqrt(np.mean(e ** 2))
    mape = 100 * np.mean(np.abs(e / y)); r2 = 1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2)
    return dict(acc=100 - mape, mae=mae, rmse=rmse, mape=mape, r2=r2)

def dm_test(y, p1, p2):
    """Diebold-Mariano, squared-error loss, h=1, Harvey-Leybourne-Newbold small-sample correction.
    H0: equal accuracy. Negative stat => model 1 has smaller loss than model 2."""
    d = (y - p1) ** 2 - (y - p2) ** 2; n = len(d)
    dm = d.mean() / math.sqrt(d.var(ddof=0) / n)
    dm *= math.sqrt((n - 1) / n)
    return float(dm), float(2 * (1 - stats.t.cdf(abs(dm), df=n - 1)))

## 4. One stock end to end

In [ ]:
def run_stock(code, path, verbose=True):
    t0 = time.time()
    d = add_indicators(load_csv(path))
    X, y, dates, last_close, tr, te, inv = make_samples(d)
    Xtr, ytr = X[tr], y[tr]; Xte, yte = X[te], y[te]
    if len(Xtr) < 300 or len(Xte) == 0:
        print(f"[{code}] skipped (train={len(Xtr)}, test={len(Xte)})"); return None
    print(f"[{code}] train samples={len(Xtr)}  test sessions={len(Xte)}  device={DEVICE}  fast={FAST}")

    # --- out-of-fold predictions for the meta-learner (chronological, expanding window) ---
    oof = {"lstm": [], "qk": []}; oof_y = []
    for f, (a, b) in enumerate(TimeSeriesSplit(n_splits=N_SPLITS).split(Xtr)):
        for kind in ("lstm", "qk"):
            net, _ = fit(kind, Xtr[a], ytr[a]); oof["qk" if kind == "qk" else "lstm"].append(predict(net, Xtr[b]))
        oof_y.append(ytr[b]); print(f"   fold {f+1}/{N_SPLITS} done  ({time.time()-t0:.0f}s)")
    Z = np.column_stack([np.concatenate(oof["lstm"]), np.concatenate(oof["qk"])]); zy = np.concatenate(oof_y)
    meta = Ridge(alpha=1.0).fit(Z, zy)

    # --- final base learners on the full training window ---
    tq = time.time()
    lstm, ep_l = fit("lstm", Xtr, ytr, verbose=False)
    qk, ep_q = fit("qk", Xtr, ytr, verbose=False)
    train_time = time.time() - tq
    p_l, p_q = predict(lstm, Xte), predict(qk, Xte)
    p_e = meta.predict(np.column_stack([p_l, p_q]))

    # --- back to price units ---
    actual = inv(yte); P = {"classical": inv(p_l), "qklstm": inv(p_q), "ensemble": inv(p_e), "naive": last_close[te]}
    M = {k: metrics(actual, v) for k, v in P.items()}
    dm = {}
    for name, (a_, b_) in {"ens_vs_classical": ("ensemble", "classical"), "ens_vs_qk": ("ensemble", "qklstm"),
                           "ens_vs_naive": ("ensemble", "naive"), "classical_vs_naive": ("classical", "naive")}.items():
        s, p = dm_test(actual, P[a_], P[b_]); dm[name] = dict(stat=s, p=p)
    res = dict(code=code, name=NAMES.get(code, code), unit="thousand VND" if actual.mean() < 1000 else "index points",
               dates=[x.strftime("%Y-%m-%d") for x in dates[te]], actual=actual.round(4).tolist(),
               **{k: v.round(4).tolist() for k, v in P.items()},
               metrics={k: {m: round(float(x), 4) for m, x in v.items()} for k, v in M.items()},
               dm=dm, ridge=dict(w_classical=float(meta.coef_[0]), w_qk=float(meta.coef_[1]), intercept=float(meta.intercept_)),
               config=dict(seq=SEQ, units=UNITS, act=ACT, qubits=N_QUBITS, n_ref=N_REF, q_layers=Q_LAYERS, max_epochs=MAX_EPOCHS, batch_qk=BATCH_QK,
                           patience=PATIENCE, n_splits=N_SPLITS, fast=FAST, epochs_lstm=ep_l, epochs_qk=ep_q,
                           params_lstm=n_params(lstm), params_qk=n_params(qk), train_seconds=round(train_time, 1),
                           train_samples=int(len(Xtr)), device=DEVICE, seed=SEED))
    if verbose:
        for k, v in M.items(): print(f"   {k:10s} MAE={v['mae']:.4f} RMSE={v['rmse']:.4f} MAPE={v['mape']:.2f}% R2={v['r2']:.3f}")
        print(f"   ridge w=({meta.coef_[0]:.2f},{meta.coef_[1]:.2f})  DM ens-vs-classical p={dm['ens_vs_classical']['p']:.3f}  "
              f"ens-vs-naive p={dm['ens_vs_naive']['p']:.3f}  total={time.time()-t0:.0f}s")
    return res

def export(res):
    os.makedirs(OUT_DIR, exist_ok=True)
    js = json.dumps(res, separators=(",", ":"))
    open(f"{OUT_DIR}/{res['code']}.json", "w").write(js)
    open(f"{OUT_DIR}/{res['code']}.js", "w").write(f'window.RESULTS=window.RESULTS||{{}};window.RESULTS["{res["code"]}"]={js};')
    codes = sorted(os.path.basename(p)[:-3] for p in glob.glob(f"{OUT_DIR}/*.js") if not p.endswith("manifest.js"))
    open(f"{OUT_DIR}/manifest.js", "w").write("window.MANIFEST=" + json.dumps(codes) + ";")

## 5. Simulator sanity check against PennyLane (optional: `pip install pennylane`)

In [ ]:
def check_against_pennylane(n_samples=3):
    import pennylane as qml
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def circ(ang):
        for q in range(N_QUBITS): qml.Hadamard(q)
        for _ in range(Q_LAYERS):
            for q in range(N_QUBITS): qml.RZ(ang[N_QUBITS + q], wires=q); qml.RY(ang[q], wires=q)
            for q in range(N_QUBITS - 1): qml.CNOT(wires=[q, q + 1])
        return qml.state()
    sim = QSim(); worst = 0
    for _ in range(n_samples):
        a = (torch.rand(1, 2 * N_QUBITS) * 2 * math.pi - math.pi)
        mine = sim(a)[0].detach().numpy(); ref = np.array(circ(a[0].numpy()))
        worst = max(worst, 1 - abs(np.vdot(ref, mine)) ** 2)          # global phase irrelevant
    print(f"PennyLane vs torch simulator: worst infidelity = {worst:.2e}")
    return worst

check_against_pennylane()

## 6. Run everything

In [ ]:
if __name__ == "__main__" or "get_ipython" in globals():
    files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
    only = os.environ.get("QK_ONLY")                        # e.g. QK_ONLY=BID,VNINDEX
    if only: files = [f for f in files if os.path.basename(f)[:-4] in only.split(",")]
    print(f"{len(files)} CSV file(s) in {DATA_DIR}")
    for f in files:
        code = os.path.basename(f)[:-4]
        r = run_stock(code, f)
        if r: export(r)